In [15]:
import sympy as sp
import numpy as np
import math

def solve_nonlinear_system(G_funcs, variables, D, X0, epsilon, precision):
    # ==========================================
    # HÀM PHỤ TRỢ: CHỐNG TRÒN SỐ VỀ 0
    # ==========================================
    def fmt_err(val, p):
        if val == 0:
            return f"0.{'0'*p}"
        # Nếu giá trị quá nhỏ (nhỏ hơn ngưỡng hiển thị), bật ký hiệu khoa học
        if abs(val) < 10**(-p):
            m, e = f"{val:.2e}".split('e')
            return f"{m} \\times 10^{{{int(e)}}}"
        return f"{val:.{p}f}"

    n_vars = len(variables)
    
    # 1. TÍNH MA TRẬN JACOBIAN
    J = sp.Matrix(G_funcs).jacobian(variables)
    
    grids = [np.linspace(d[0], d[1], 10) for d in D]
    mesh = np.array(np.meshgrid(*grids)).T.reshape(-1, n_vars)

    # ==========================================
    # OUTPUT FORMAT 
    # ==========================================
    print("### 1.1.1 Chứng minh điều kiện hội tụ\n")
    print("**Hệ hàm lặp $X=G(X)$:**")
    G_latex = r" \\ ".join([sp.latex(g) for g in G_funcs])
    print(f"$$G(X) = \\begin{{bmatrix}} {G_latex} \\end{{bmatrix}}$$")
    
    print(f"\n**Khảo sát trên miền $D$:**")
    for i, var in enumerate(variables):
        print(f"* ${var} \\in [{D[i][0]}, {D[i][1]}]$")
        
    # Tính giá trị min/max của từng hàm g_i(X) trên lưới điểm mesh
    is_subset = True
    for i in range(n_vars):
        # Tạo lambda riêng cho từng hàm g_i để tránh lỗi mảng khi có hàm hằng số
        g_i_lambda = sp.lambdify(variables, G_funcs[i], 'numpy')
        
        # Đánh giá hàm trên toàn bộ các điểm thuộc lưới
        vals = [float(g_i_lambda(*pt)) for pt in mesh]
        g_min = min(vals)
        g_max = max(vals)
        
        domain_min, domain_max = D[i]
        
        # Kiểm tra xem khoảng giá trị của g_i có nằm gọn trong D[i] không
        if g_min >= domain_min and g_max <= domain_max:
            subset_str = f"\\subset [{domain_min}, {domain_max}]"
        else:
            subset_str = f"\\not\\subset [{domain_min}, {domain_max}]"
            is_subset = False
            
        print(f"* $g_{i+1}(X) \\in [{g_min:.{precision}f}, {g_max:.{precision}f}] {subset_str}$")

    if is_subset:
        print("* $\\Rightarrow G(D) \\subset D$")
    else:
        print("* $\\Rightarrow G(D) \\not\\subset D$. Cảnh báo: Điều kiện ánh xạ vào trong không thỏa mãn!")
    
    print("\n**Ma trận Jacobian $J(X)$:**")
    J_latex = sp.latex(J).replace('matrix', 'bmatrix')
    print(f"$$J(X) = {J_latex}$$")
    
    # ĐÁNH GIÁ CHUẨN HÀNG CHI TIẾT
    print("\n**Đánh giá chuẩn hàng trên D:**")
    row_maxes = []
    
    for i in range(n_vars):
        row_expr = sum([sp.Abs(J[i, j]) for j in range(n_vars)])
        row_lambda = sp.lambdify(variables, row_expr, 'numpy')
        max_val_row = 0
        for pt in mesh:
            val = np.array(row_lambda(*pt))
            max_val_row = max(max_val_row, float(val))
            
        row_maxes.append(max_val_row)
        expr_latex = sp.latex(row_expr)
        print(f"* Hàng {i+1}: $\\Sigma |J_{{{i+1}j}}| = {expr_latex} \\le {max_val_row:.{precision}f}$")
        
    q = max(row_maxes)
    
    print(f"\nHệ số co $q = \\max_{{X \\in D}} ||J(X)||_{{\\infty}} \\approx {q:.{precision}f}$.")
    if q < 1:
        print("Do $q < 1$ và $G(D) \\subset D$ hệ thỏa mãn điều kiện hội tụ của phương pháp lặp đơn trên miền D.")
    else:
        print("Cảnh báo: $q \\ge 1$. Phương pháp có thể không hội tụ trên miền D.")
        return

    print("\n### 1.1.2 Thuật toán tìm nghiệm\n")
    print("**Công thức lặp:**")
    print("$$X^{(k)} = G(X^{(k-1)})$$")
    
    X_0 = np.array(X0, dtype=float)
    X0_str = ", ".join([f"{x:.{precision}f}" for x in X_0])
    print(f"**Chọn vector xấp xỉ đầu:** $X_0 = [{X0_str}]^T$.")
    
    G_lambda = sp.lambdify(variables, G_funcs, 'numpy')
    
    X_history = [X_0]
    Delta_history = [0.0]
    
    # Tính X_1
    X_1 = np.array(G_lambda(*X_0)).flatten()
    norm_01 = np.max(np.abs(X_1 - X_0))
    X_history.append(X_1)
    Delta_history.append(norm_01)
    
    # ------------------------------------------
    # 1.1.2.1 Tiên nghiệm (Dự đoán k)
    # ------------------------------------------
    print("\n**1.1.2.1 Dự đoán số bước lặp (Sai số tiên nghiệm)**")
    X1_str = ", ".join([f"{x:.{precision}f}" for x in X_1])
    print(f"Tính bước lặp đầu tiên:\n$X_1 = G(X_0) \\approx [{X1_str}]^T$")
    print(f"Khoảng cách cơ sở: $||X_1 - X_0||_{{\\infty}} = {fmt_err(norm_01, precision)}$.")
    
    target_val = (epsilon * (1 - q)) / norm_01
    K_target = math.ceil(math.log(target_val) / math.log(q))
    print(f"Để đạt sai số $\\epsilon = {epsilon}$, số bước lặp $k$ tối đa cần thực hiện là:")
    print(f"$$\\frac{{q^k}}{{1-q}}||X_1 - X_0||_{{\\infty}} \\le {epsilon} \\Rightarrow k \\ge \\frac{{\\ln\\left(\\frac{{{epsilon}(1-q)}}{{||X_1 - X_0||_{{\\infty}}}}\\right)}}{{\\ln(q)}} \\approx {math.log(target_val) / math.log(q):.1f} \\Rightarrow k = {K_target}$$")
    
    # ------------------------------------------
    # 1.1.2.2 Hậu nghiệm (Quá trình lặp thực tế)
    # ------------------------------------------
    print("\n**1.1.2.2 Chi tiết quá trình lặp (Sai số hậu nghiệm)**")
    factor = q / (1 - q)
    print(f"Điều kiện dừng: $Err_{{pos}} = \\frac{{q}}{{1-q}}||X_k - X_{{k-1}}||_{{\\infty}} \\le {epsilon}$. Với $\\frac{{q}}{{1-q}} \\approx {factor:.4f}$.")
    
    k = 1
    err_pos = factor * norm_01
    
    print(f"\n* Bước $k=1$: $X_1 \\approx [{X1_str}]^T$")
    print(f"  * $\\Delta_1 = {fmt_err(norm_01, precision)}$")
    print(f"  * $Err_1 = {factor:.4f} \\times \\Delta_1 = {fmt_err(err_pos, precision)} > {epsilon}$")
    
    while err_pos > epsilon:
        k += 1
        X_next = np.array(G_lambda(*X_history[-1])).flatten()
        delta_k = np.max(np.abs(X_next - X_history[-1]))
        err_pos = factor * delta_k
        
        X_history.append(X_next)
        Delta_history.append(delta_k)
        
        if k > 100:
            print("Thuật toán dừng sớm: Không hội tụ sau 100 bước.")
            break
            
    Xk_str = ", ".join([f"{x:.{precision}f}" for x in X_history[-1]])
    print(f"\n* Bước cuối cùng ($k={k}$): $X_{{{k}}} \\approx [{Xk_str}]^T$")
    print(f"  * $\\Delta_{{{k}}} = ||X_{{{k}}} - X_{{{k-1}}}||_{{\\infty}} = {fmt_err(Delta_history[-1], precision)}$")
    print(f"  * $Err_{{{k}}} = {factor:.4f} \\times {fmt_err(Delta_history[-1], precision)} = {fmt_err(err_pos, precision)} \\le {epsilon}$ (Thỏa mãn).")
    
    print(f"\n**Nghiệm gần đúng:** $X \\approx [{Xk_str}]^T$")
    
    # ------------------------------------------
    # 1.1.3 Bảng tổng hợp
    # ------------------------------------------
    print("\n### 1.1.3 Bảng tổng hợp quá trình lặp\n")
    
    headers = ["$k$"] + [f"$x_{i+1}^{{(k)}}$" for i in range(n_vars)] + ["$\\Delta_k$"]
    header_str = " | ".join(headers)
    print(f"| {header_str} |")
    print("|" + "|".join(["---"] * len(headers)) + "|")
    
    def print_row(idx):
        row_data = [str(idx)]
        row_data.extend([f"{x:.{precision}f}" for x in X_history[idx]])
        if idx == 0:
            row_data.append(f"${0.0:.{precision}f}$")
        else:
            # Gói kết quả trả về của hàm fmt_err vào biểu thức $ $ cho Markdown
            row_data.append(f"${fmt_err(Delta_history[idx], precision)}$")
        print("| " + " | ".join(row_data) + " |")

    n_steps = len(X_history)
    
    for i in range(min(3, n_steps)):
        print_row(i)
        
    if n_steps > 5:
        print("| ... | " + " | ".join(["..."] * n_vars) + " | ... |")
        
    if n_steps > 3:
        for i in range(max(3, n_steps - 2), n_steps):
            print_row(i)

if __name__ == "__main__":
    print("=== CHƯƠNG TRÌNH GIẢI HỆ PHƯƠNG TRÌNH PHI TUYẾN ===")
    try:
        n_vars = int(input("Nhập số lượng biến của hệ (ví dụ: 3): "))
        
        var_input = input("Nhập tên các biến, cách nhau bởi dấu cách (ví dụ: x1 x2 x3): ")
        var_names = var_input.split()
        variables = sp.symbols(var_names)
        
        G_funcs = []
        for i in range(n_vars):
            raw_func = input(f"Nhập hàm g_{i+1}(X): ")
            G_funcs.append(sp.sympify(raw_func.replace('^', '**')))
            
        D = []
        for i in range(n_vars):
            d_input = input(f"Nhập miền D cho biến {var_names[i]} dưới dạng 'a b' (ví dụ: -1 1 hoặc -1/3 1/3): ")
            # Dùng sympy để xử lý trước khi ép sang float, giúp hỗ trợ phân số và biểu thức
            parts = d_input.split()
            a = float(sp.sympify(parts[0]))
            b = float(sp.sympify(parts[1]))
            D.append((a, b))
            
        x0_input = input("Nhập vector giá trị khởi tạo X0, cách nhau bởi dấu cách (ví dụ: 0 0 0 hoặc 1/2 1/3 1/4): ")
        # Tương tự, hỗ trợ nhập X0 dưới dạng phân số
        X_init = [float(sp.sympify(val)) for val in x0_input.split()]
        
        epsilon = float(input("Nhập sai số dung sai epsilon (ví dụ: 1e-6): "))
        precision = int(input("Nhập số chữ số thập phân làm tròn (ví dụ: 7): "))
        
        print("\n" + "="*50 + "\n")
        solve_nonlinear_system(
            G_funcs=G_funcs, 
            variables=variables, 
            D=D, 
            X0=X_init, 
            epsilon=epsilon, 
            precision=precision
        )
        
    except Exception as e:
        print(f"\nLỗi: {e}. Vui lòng kiểm tra lại định dạng dữ liệu đầu vào.")

=== CHƯƠNG TRÌNH GIẢI HỆ PHƯƠNG TRÌNH PHI TUYẾN ===




### 1.1.1 Chứng minh điều kiện hội tụ

**Hệ hàm lặp $X=G(X)$:**
$$G(X) = \begin{bmatrix} \frac{\sin{\left(x y \right)}}{3} \\ \frac{\cos{\left(x^{2} + y^{2} \right)}}{4} \end{bmatrix}$$

**Khảo sát trên miền $D$:**
* $x \in [0.0, 0.3333333333333333]$
* $y \in [0.0, 0.25]$
* $g_1(X) \in [0.0000000, 0.0277456] \subset [0.0, 0.3333333333333333]$
* $g_2(X) \in [0.2462419, 0.2500000] \subset [0.0, 0.25]$
* $\Rightarrow G(D) \subset D$

**Ma trận Jacobian $J(X)$:**
$$J(X) = \left[\begin{bmatrix}\frac{y \cos{\left(x y \right)}}{3} & \frac{x \cos{\left(x y \right)}}{3}\\- \frac{x \sin{\left(x^{2} + y^{2} \right)}}{2} & - \frac{y \sin{\left(x^{2} + y^{2} \right)}}{2}\end{bmatrix}\right]$$

**Đánh giá chuẩn hàng trên D:**
* Hàng 1: $\Sigma |J_{1j}| = \frac{\left|{x \cos{\left(x y \right)}}\right|}{3} + \frac{\left|{y \cos{\left(x y \right)}}\right|}{3} \le 0.1937697$
* Hàng 2: $\Sigma |J_{2j}| = \frac{\left|{x \sin{\left(x^{2} + y^{2} \right)}}\right|}{2} + \frac{\left|{y \sin{\left(x^{2} + y^